# Parallelise CPU work across processes

**The question.** You have a CPU-bound function — it crunches numbers, parses, hashes, resizes images — and a big pile of inputs. It pins one core while the other seven sit idle. You want to use them all.

The answer: `concurrent.futures.ProcessPoolExecutor`. Each worker is a separate interpreter with its own GIL, so the work runs in true parallel across cores. The interface matches the thread pool; the differences are that the function must be top-level (picklable), the launch must be guarded by `if __name__ == "__main__"`, and you should chunk small tasks to beat the overhead.

> Spawns real processes — save as a `.py` file and run it locally; it won't run in the browser or a bare REPL.

## The pattern

Define the worker at module top level, create the pool inside the `__main__` guard, and `map` your inputs across it.

In [ ]:
# save as primes.py and run:  python primes.py
from concurrent.futures import ProcessPoolExecutor
import time

def is_prime(n):                      # top-level so it can be pickled to workers
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

def main():
    numbers = range(2, 200_000)
    start = time.perf_counter()
    with ProcessPoolExecutor() as pool:           # one worker per core by default
        flags = pool.map(is_prime, numbers, chunksize=2000)
        count = sum(flags)
    print(f'{count} primes in {time.perf_counter() - start:.2f}s')

if __name__ == '__main__':            # mandatory: stops workers re-running main()
    main()

## Why `chunksize` matters

Sending each number to a worker individually means pickling and shipping 200,000 tiny messages — the overhead swamps the actual work. `chunksize=2000` hands each worker a batch of 2000 inputs at a time, cutting the messaging cost by three orders of magnitude. For large iterables of cheap-to-medium tasks, tuning `chunksize` is often a bigger win than adding cores.

A rough guide: aim for each chunk to take tens of milliseconds or more of compute. Too small and overhead dominates; too large and the cores finish unevenly (one worker stuck on the last fat chunk while others idle).

## Returning richer results

`map` returns whatever the worker returns, in input order. To keep the input alongside the result, return a tuple — and remember the return value must be picklable too.

In [ ]:
def classify(n):
    return (n, is_prime(n))           # tuple of (input, result)

def main():
    with ProcessPoolExecutor() as pool:
        for n, prime in pool.map(classify, range(2, 12)):
            print(f'{n}: {"prime" if prime else "composite"}')

if __name__ == '__main__':
    main()

## Measure the speedup honestly

Process pools have real startup and pickling overhead, and Amdahl's law caps the gain at the parallelisable fraction of your program. Always time both ways on the *actual* workload before declaring victory — for small inputs the sequential version often wins.

In [ ]:
def main():
    numbers = range(2, 200_000)

    start = time.perf_counter()
    seq = sum(is_prime(n) for n in numbers)
    seq_time = time.perf_counter() - start

    start = time.perf_counter()
    with ProcessPoolExecutor() as pool:
        par = sum(pool.map(is_prime, numbers, chunksize=2000))
    par_time = time.perf_counter() - start

    print(f'sequential {seq_time:.2f}s | parallel {par_time:.2f}s | {seq_time/par_time:.1f}x')

if __name__ == '__main__':
    main()

## Gotchas to expect

- **`PicklingError` / `AttributeError` on a lambda or local function** — the worker function must be importable at module top level.
- **Programs that spawn endlessly or error on start** — you forgot the `if __name__ == "__main__"` guard.
- **No speedup, or a slowdown** — tasks are too small (chunk them), the work isn't really CPU-bound (use threads/async), or large data is being copied to and from workers.

For the lower-level `multiprocessing` primitives (shared `Value`/`Array`, `Queue`, `Pool`), see the [threading and multiprocessing reference](../reference/threading-and-multiprocessing-reference.md).